In [1]:
import os
import re
import glob
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

# =============================================================================
# User settings
# =============================================================================
RRS_TIF = "/projectnb/nasa-marsh/Hangjie/Louisiana/original_monthly_product/cci_Rrs665_monthly_GoM.tif"
BAND_MAP_TXT = "/projectnb/nasa-marsh/Hangjie/Louisiana/original_monthly_product/cci_Rrs665_monthly_GoM_band2date.txt"

# Set this to your 4 m isobath section vector file if you want to bypass auto-search.
# Supported examples: .shp, .gpkg, .geojson
SECTION_VECTOR = "/projectnb/nasa-marsh/Hangjie/Louisiana/bathymetry/4m_isobath_20260423.shp"

# If your section file already contains a field identifying section IDs/names,
# set it here (for example: "section", "Section", "name", "id").
# Leave as None to auto-detect, or to assign Section-1..Section-4 by feature order.
SECTION_FIELD = None

OUTPUT_DIR = "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/SSC"

# Ocean Colour CCI nodata
NODATA_VALUE = -9999.0

# By default, only nodata and non-finite values are excluded.
# If you also want to exclude negative Rrs values, set this to 0.0.
RRS_MIN = None

# =============================================================================
# Figure tuning parameters
# =============================================================================
# Global left-side y-axis label position in figure coordinates.
# Increase this value to move the label rightward (closer to tick labels).
# Decrease this value to move the label leftward.
SSC_LABEL_X = 0.10

# Font size of the global left-side y-axis label
SSC_LABEL_FONTSIZE = 13

# Padding for the left time-series y-limits
TS_Y_PAD_FRAC = 0.08

# Padding for the right climatology y-limits
# Use a smaller padding to make seasonal amplitude more visually distinct.
CLIM_Y_PAD_FRAC = 0.05

# =============================================================================
# SSC algorithm
# =============================================================================
def rrs665_to_ssc(rrs):
    """
    Convert Rrs665 to SSC using the user-provided algorithm.

    SSC = 33620 * Rrs665^2 + 1473 * Rrs665 + 0.004
    """
    return 33620.0 * rrs * rrs + 1473.0 * rrs + 0.004


# =============================================================================
# Helper functions
# =============================================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def parse_band_mapping(txt_file):
    """
    Read the band-to-date mapping text file.

    Expected columns:
    BandIndex, ISODate(YYYY-MM-DD), Year, Month, Day, SourceFileName
    """
    records = []

    with open(txt_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if (not line) or line.startswith("#"):
                continue

            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 6:
                continue

            band_index = int(parts[0])
            iso_date = pd.to_datetime(parts[1])
            year = int(parts[2])
            month = int(parts[3])
            day = int(parts[4])
            source_name = ",".join(parts[5:]).strip()

            records.append(
                {
                    "band": band_index,
                    "date": iso_date,
                    "year": year,
                    "month": month,
                    "day": day,
                    "source_file": source_name,
                }
            )

    df = pd.DataFrame(records).sort_values("band").reset_index(drop=True)

    if df.empty:
        raise ValueError(f"No valid band-date records were found in: {txt_file}")

    return df


def auto_find_section_vector(search_root):
    """
    Try to find a likely 4 m section vector automatically.
    """
    patterns = [
        "**/*4m*section*.shp",
        "**/*4m*section*.gpkg",
        "**/*4m*section*.geojson",
        "**/*section*4m*.shp",
        "**/*section*4m*.gpkg",
        "**/*section*4m*.geojson",
        "**/*4m*.shp",
        "**/*4m*.gpkg",
        "**/*4m*.geojson",
    ]

    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(os.path.join(search_root, pattern), recursive=True))

    candidates = sorted(set(candidates))

    if len(candidates) == 0:
        raise FileNotFoundError(
            "No likely 4 m section vector file was found automatically. "
            "Please set SECTION_VECTOR explicitly."
        )

    if len(candidates) == 1:
        return candidates[0]

    print("Auto-search found multiple candidate section files:")
    for i, c in enumerate(candidates, start=1):
        print(f"  {i:02d}. {c}")

    raise RuntimeError(
        "Multiple candidate section files were found. "
        "Please set SECTION_VECTOR explicitly."
    )


def detect_section_field(gdf, preferred=None):
    """
    Detect the field storing section identifiers.
    """
    if preferred is not None:
        if preferred in gdf.columns:
            return preferred
        raise KeyError(f"SECTION_FIELD='{preferred}' was not found in the section file.")

    candidates = [
        "section", "Section", "SECTION",
        "section_id", "Section_ID", "SECTION_ID",
        "name", "Name", "NAME",
        "id", "ID",
        "label", "Label", "LABEL",
        "seg", "segment", "Segment"
    ]

    for c in candidates:
        if c in gdf.columns:
            return c

    return None


def normalize_section_label(value, fallback_idx=None):
    """
    Convert any section identifier to a standard label such as Section-1.
    """
    if pd.isna(value):
        if fallback_idx is None:
            return None
        return f"Section-{fallback_idx}"

    text = str(value).strip()

    if text == "":
        if fallback_idx is None:
            return None
        return f"Section-{fallback_idx}"

    m = re.search(r"(\d+)", text)
    if m:
        return f"Section-{int(m.group(1))}"

    if fallback_idx is not None:
        return f"Section-{fallback_idx}"

    return text


def prepare_section_geometries(section_vector, raster_crs, section_field=None):
    """
    Read the section vector, reproject to raster CRS, and return one dissolved
    geometry per section.
    """
    gdf = gpd.read_file(section_vector)

    if gdf.empty:
        raise ValueError(f"No features were found in section vector: {section_vector}")

    if gdf.crs is None:
        raise ValueError(
            "The section vector has no CRS defined. "
            "Please assign the correct CRS before running."
        )

    gdf = gdf.to_crs(raster_crs).copy()
    gdf = gdf[gdf.geometry.notnull()].copy()

    chosen_field = detect_section_field(gdf, preferred=section_field)

    if chosen_field is not None:
        gdf["section_label"] = [
            normalize_section_label(v, fallback_idx=i + 1)
            for i, v in enumerate(gdf[chosen_field].tolist())
        ]
    else:
        gdf["section_label"] = [f"Section-{i + 1}" for i in range(len(gdf))]

    dissolved = gdf[["section_label", "geometry"]].dissolve(
        by="section_label", as_index=False
    )

    def section_sort_key(label):
        m = re.search(r"(\d+)", str(label))
        return int(m.group(1)) if m else 9999

    dissolved = dissolved.sort_values(
        by="section_label",
        key=lambda s: s.map(section_sort_key)
    ).reset_index(drop=True)

    if len(dissolved) != 4:
        warnings.warn(
            f"The section file produced {len(dissolved)} dissolved section(s), "
            f"not 4. Please verify SECTION_FIELD and the input file."
        )

    return dissolved, chosen_field


def build_masks(section_gdf, out_shape, transform):
    """
    Build a boolean mask for each section and for the union of all sections.
    True means the pixel belongs to the section.

    all_touched=True is used so that every raster pixel intersected by the
    section geometry is included.
    """
    masks = {}

    for _, row in section_gdf.iterrows():
        geom = row.geometry
        label = row["section_label"]

        mask = geometry_mask(
            geometries=[geom],
            out_shape=out_shape,
            transform=transform,
            invert=True,
            all_touched=True,
        )
        masks[label] = mask

    union_mask = np.zeros(out_shape, dtype=bool)
    for mask in masks.values():
        union_mask |= mask

    masks["Entire Section"] = union_mask
    return masks


def compute_monthly_section_means(rrs_tif, band_df, masks, nodata_value=-9999.0, rrs_min=None):
    """
    For each monthly band, compute mean SSC for each section and valid pixel counts.
    """
    results = []
    valid_counts = []

    with rasterio.open(rrs_tif) as src:
        if src.count != len(band_df):
            raise ValueError(
                f"Band count mismatch: raster has {src.count} bands, "
                f"but mapping file has {len(band_df)} records."
            )

        for _, rec in band_df.iterrows():
            band = int(rec["band"])
            arr = src.read(band).astype(np.float32)

            valid = np.isfinite(arr) & (arr != nodata_value)
            if rrs_min is not None:
                valid &= arr >= rrs_min

            row_data = {
                "date": rec["date"],
                "year": rec["year"],
                "month": rec["month"],
                "band": band,
                "source_file": rec["source_file"],
            }

            row_counts = {
                "date": rec["date"],
                "year": rec["year"],
                "month": rec["month"],
                "band": band,
            }

            if valid.any():
                ssc = np.full(arr.shape, np.nan, dtype=np.float32)
                ssc[valid] = rrs665_to_ssc(arr[valid])
            else:
                ssc = np.full(arr.shape, np.nan, dtype=np.float32)

            for label, mask in masks.items():
                section_valid = valid & mask
                n_valid = int(section_valid.sum())
                row_counts[label] = n_valid

                if n_valid > 0:
                    row_data[label] = float(np.nanmean(ssc[section_valid]))
                else:
                    row_data[label] = np.nan

            results.append(row_data)
            valid_counts.append(row_counts)

    ts_df = pd.DataFrame(results).sort_values("date").reset_index(drop=True)
    count_df = pd.DataFrame(valid_counts).sort_values("date").reset_index(drop=True)

    return ts_df, count_df


def compute_monthly_climatology(ts_df, section_columns):
    """
    Compute monthly climatology statistics for each section.
    """
    clim_long = []

    for label in section_columns:
        grouped = ts_df.groupby("month")[label]

        stats = pd.DataFrame(
            {
                "mean": grouped.mean(),
                "std": grouped.std(),
                "median": grouped.median(),
                "p25": grouped.quantile(0.25),
                "p75": grouped.quantile(0.75),
                "min": grouped.min(),
                "max": grouped.max(),
                "count": grouped.count(),
            }
        ).reset_index()

        stats["section"] = label
        clim_long.append(stats)

    clim_long = pd.concat(clim_long, ignore_index=True)

    clim_wide_mean = clim_long.pivot(index="month", columns="section", values="mean").reset_index()
    clim_wide_std = clim_long.pivot(index="month", columns="section", values="std").reset_index()

    return clim_long, clim_wide_mean, clim_wide_std


def compute_mask_pixel_counts(masks):
    """
    Count total touched pixels in each section mask.
    """
    rows = []
    for label, mask in masks.items():
        rows.append(
            {
                "section": label,
                "mask_pixel_count": int(mask.sum()),
            }
        )
    return pd.DataFrame(rows)


def compute_axis_limits(values, pad_frac=0.08, default=(0.0, 1.0)):
    """
    Compute dynamic y-axis limits from a 1D array-like object.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return default

    ymin = np.nanmin(values)
    ymax = np.nanmax(values)
    yr = ymax - ymin

    if yr > 0:
        pad = pad_frac * yr
    else:
        pad = max(0.5, 0.08 * abs(ymax) if ymax != 0 else 1.0)

    return ymin - pad, ymax + pad


def make_section_plot(ts_df, clim_long, output_png):
    """
    Create a figure with long monthly time series panels on the left and
    monthly climatology panels on the right.

    The climatology panel uses its own tighter y-limits to better show
    seasonal amplitude.
    """
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.size": 11,
            "axes.linewidth": 1.0,
            "xtick.direction": "out",
            "ytick.direction": "out",
        }
    )

    section_order = ["Section-1", "Section-2", "Section-3", "Section-4", "Entire Section"]
    color_map = {
        "Section-1": "#1f51ff",
        "Section-2": "#78c83c",
        "Section-3": "#f0a11b",
        "Section-4": "#e41a1c",
        "Entire Section": "#222222",
    }

    section_order = [s for s in section_order if s in ts_df.columns]

    fig = plt.figure(figsize=(17, 8), dpi=300)
    gs = GridSpec(
        nrows=len(section_order),
        ncols=2,
        width_ratios=[12.5, 2.0],
        hspace=0.16,
        wspace=0.08,
        figure=fig,
    )

    dates = pd.to_datetime(ts_df["date"])
    years = np.arange(dates.dt.year.min(), dates.dt.year.max() + 1)

    left_axes = []
    right_axes = []

    for i, label in enumerate(section_order):
        ax_ts = fig.add_subplot(gs[i, 0])
        ax_clim = fig.add_subplot(gs[i, 1])
        left_axes.append(ax_ts)
        right_axes.append(ax_clim)

        color = color_map.get(label, "k")
        y = ts_df[label].to_numpy(dtype=float)

        clim_sub = clim_long[clim_long["section"] == label].sort_values("month")

        # ---------------------------------------------------------------------
        # Dynamic y-limits for the long monthly time series
        # Keep this broader so the full temporal variability is visible.
        # ---------------------------------------------------------------------
        ts_candidates = y[np.isfinite(y)]
        ts_ymin, ts_ymax = compute_axis_limits(
            ts_candidates,
            pad_frac=TS_Y_PAD_FRAC,
            default=(0.0, 1.0),
        )

        # ---------------------------------------------------------------------
        # Dynamic y-limits for climatology only
        # Use a tighter range based only on the climatological curve and IQR.
        # This makes seasonal amplitude clearer.
        # ---------------------------------------------------------------------
        clim_candidates = np.concatenate(
            [
                clim_sub["mean"].to_numpy(dtype=float),
                clim_sub["p25"].to_numpy(dtype=float),
                clim_sub["p75"].to_numpy(dtype=float),
            ]
        )
        clim_ymin, clim_ymax = compute_axis_limits(
            clim_candidates,
            pad_frac=CLIM_Y_PAD_FRAC,
            default=(0.0, 1.0),
        )

        # ---------------------------------------------------------------------
        # Left panel: monthly time series
        # ---------------------------------------------------------------------
        ax_ts.set_facecolor("#f2f2f2")

        for yr in years[1:]:
            ax_ts.axvline(
                pd.Timestamp(f"{yr}-01-01"),
                color="0.55",
                lw=0.9,
                ls=(0, (3, 4)),
                zorder=0,
            )

        ax_ts.plot(
            dates,
            y,
            color=color,
            lw=1.4,
            marker="o",
            ms=3.2,
            mfc="white",
            mec=color,
            mew=1.1,
            zorder=3,
        )

        ax_ts.set_ylim(ts_ymin, ts_ymax)
        ax_ts.tick_params(axis="y", right=True, labelright=True, length=4)

        ax_ts.text(
            0.60,
            0.84,
            f"{label} SSC",
            color=color,
            transform=ax_ts.transAxes,
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
        )

        if i < len(section_order) - 1:
            ax_ts.tick_params(axis="x", labelbottom=False)
        else:
            ax_ts.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
            ax_ts.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

        if i == 0:
            secax = ax_ts.secondary_xaxis("top")
            midyear_ticks = [pd.Timestamp(f"{yr}-07-01") for yr in years]
            secax.set_xticks(midyear_ticks)
            secax.set_xticklabels([str(yr) for yr in years])
            secax.tick_params(length=0, pad=6, labelsize=12)

        # ---------------------------------------------------------------------
        # Right panel: monthly climatology
        # ---------------------------------------------------------------------
        ax_clim.set_facecolor("#f2f2f2")

        months = clim_sub["month"].to_numpy(dtype=int)
        mean = clim_sub["mean"].to_numpy(dtype=float)
        p25 = clim_sub["p25"].to_numpy(dtype=float)
        p75 = clim_sub["p75"].to_numpy(dtype=float)

        ax_clim.fill_between(months, p25, p75, color=color, alpha=0.25, lw=0)
        ax_clim.plot(
            months,
            mean,
            color=color,
            lw=1.2,
            marker="o",
            ms=2.8,
            mfc="white",
            mec=color,
            mew=0.9,
        )

        ax_clim.set_xlim(1, 12)
        ax_clim.set_ylim(clim_ymin, clim_ymax)
        ax_clim.tick_params(axis="y", left=False, labelleft=False, right=True, labelright=True, length=4)

        if i < len(section_order) - 1:
            ax_clim.tick_params(axis="x", labelbottom=False)
        else:
            ax_clim.set_xticks([1, 4, 7, 10])
            ax_clim.set_xticklabels(["Jan", "Apr", "Jul", "Oct"])

        if i == 0:
            start_date = pd.to_datetime(ts_df["date"]).min().strftime("%b %Y")
            end_date = pd.to_datetime(ts_df["date"]).max().strftime("%b %Y")
            ax_clim.set_title(f"Monthly Climatology\n({start_date} - {end_date})", fontsize=12, pad=2)

    fig.text(
        SSC_LABEL_X,
        0.5,
        r"SSC (mg L$^{-1}$)",
        rotation=90,
        va="center",
        ha="center",
        fontsize=SSC_LABEL_FONTSIZE,
    )

    for ax in left_axes:
        ax.margins(x=0.003)

    plt.savefig(output_png, dpi=400, bbox_inches="tight")
    plt.close(fig)


# =============================================================================
# Main workflow
# =============================================================================
ensure_dir(OUTPUT_DIR)

if SECTION_VECTOR is None:
    SECTION_VECTOR = auto_find_section_vector("/projectnb/nasa-marsh/Hangjie/Louisiana")

print(f"Using section vector: {SECTION_VECTOR}")

band_df = parse_band_mapping(BAND_MAP_TXT)

with rasterio.open(RRS_TIF) as src:
    raster_crs = src.crs
    raster_shape = (src.height, src.width)
    raster_transform = src.transform

section_gdf, used_section_field = prepare_section_geometries(
    SECTION_VECTOR,
    raster_crs=raster_crs,
    section_field=SECTION_FIELD,
)

print(f"Section field used: {used_section_field}")

masks = build_masks(section_gdf, raster_shape, raster_transform)
mask_pixel_df = compute_mask_pixel_counts(masks)

ts_df, valid_count_df = compute_monthly_section_means(
    rrs_tif=RRS_TIF,
    band_df=band_df,
    masks=masks,
    nodata_value=NODATA_VALUE,
    rrs_min=RRS_MIN,
)

section_cols = [c for c in ["Section-1", "Section-2", "Section-3", "Section-4", "Entire Section"] if c in ts_df.columns]
clim_long, clim_wide_mean, clim_wide_std = compute_monthly_climatology(ts_df, section_cols)

# =============================================================================
# Save outputs
# =============================================================================
excel_path = os.path.join(OUTPUT_DIR, "Louisiana_section_SSC_monthly_timeseries_and_climatology.xlsx")
csv_ts_path = os.path.join(OUTPUT_DIR, "Louisiana_section_SSC_monthly_timeseries.csv")
csv_clim_path = os.path.join(OUTPUT_DIR, "Louisiana_section_SSC_monthly_climatology_long.csv")
csv_counts_path = os.path.join(OUTPUT_DIR, "Louisiana_section_SSC_valid_pixel_counts.csv")
csv_mask_counts_path = os.path.join(OUTPUT_DIR, "Louisiana_section_mask_pixel_counts.csv")
png_path = os.path.join(OUTPUT_DIR, "Louisiana_section_SSC_monthly_timeseries_and_climatology.png")

ts_df.to_csv(csv_ts_path, index=False)
clim_long.to_csv(csv_clim_path, index=False)
valid_count_df.to_csv(csv_counts_path, index=False)
mask_pixel_df.to_csv(csv_mask_counts_path, index=False)

section_label_df = section_gdf.copy()
section_label_df["geometry_wkt"] = section_label_df.geometry.to_wkt()
section_label_df = section_label_df.drop(columns="geometry")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    ts_df.to_excel(writer, sheet_name="monthly_timeseries", index=False)
    clim_long.to_excel(writer, sheet_name="climatology_long", index=False)
    clim_wide_mean.to_excel(writer, sheet_name="climatology_mean_wide", index=False)
    clim_wide_std.to_excel(writer, sheet_name="climatology_std_wide", index=False)
    valid_count_df.to_excel(writer, sheet_name="valid_pixel_counts", index=False)
    mask_pixel_df.to_excel(writer, sheet_name="mask_pixel_counts", index=False)
    section_label_df.to_excel(writer, sheet_name="section_labels", index=False)

# =============================================================================
# Make figure
# =============================================================================
make_section_plot(ts_df, clim_long, png_path)

print("\nFinished.")
print(f"Saved Excel : {excel_path}")
print(f"Saved CSV   : {csv_ts_path}")
print(f"Saved CSV   : {csv_clim_path}")
print(f"Saved CSV   : {csv_counts_path}")
print(f"Saved CSV   : {csv_mask_counts_path}")
print(f"Saved PNG   : {png_path}")

Using section vector: /projectnb/nasa-marsh/Hangjie/Louisiana/bathymetry/4m_isobath_20260423.shp
Section field used: Section

Finished.
Saved Excel : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/SSC/Louisiana_section_SSC_monthly_timeseries_and_climatology.xlsx
Saved CSV   : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/SSC/Louisiana_section_SSC_monthly_timeseries.csv
Saved CSV   : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/SSC/Louisiana_section_SSC_monthly_climatology_long.csv
Saved CSV   : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/SSC/Louisiana_section_SSC_valid_pixel_counts.csv
Saved CSV   : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/SSC/Louisiana_section_mask_pixel_counts.csv
Saved PNG   : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/SSC/Louisiana_section_SSC_monthly_timeseries_and_climatology.png
